# 🧪 Sesión 20 · Hito 2: API con Inyección de Dependencias, Auth JWT y RBAC

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿La seguridad de tu API resiste una auditoría?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/registro   {"usuario":"ana","password":"secreta"}  -> 201
POST /api/token      {"usuario":"ana","password":"secreta"}  -> 200 + token
GET  /api/perfil     Bearer <token_user>                     -> 200
GET  /api/admin/panel Bearer <token_user>                    -> 403
POST /api/token      {"usuario":"admin","password":"admin123"} -> 200 + token
GET  /api/admin/panel Bearer <token_admin>                   -> 200
GET  /api/perfil     sin token                               -> 401

RESUMEN: 7/7 casos correctos. ¡Hito 2 listo!
```

_La seguridad no es una capa al final: es una propiedad de todo el flujo._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Integrar registro, login, token, perfil y panel administrativo.
- Verificar 401, 403 y 200 en el mismo flujo.
- Demostrar que las contraseñas nunca se filtran.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] Registro sin filtrar contraseñas → HTTP 201
- [ ] Login del usuario registrado → HTTP 200
- [ ] El usuario accede a su perfil → HTTP 200
- [ ] El usuario no entra al panel admin → HTTP 403
- [ ] El administrador obtiene su token → HTTP 200
- [ ] El administrador entra al panel → HTTP 200
- [ ] Sin token no hay acceso → HTTP 401

## 🧩 El reto

Integra el pipeline de seguridad completo del Bloque 2 en una API de libros con usuarios, roles y rutas protegidas.

**Requisitos del contrato**

- El administrador precargado es admin/admin123; el usuario ana se registra en el flujo.
- Registro de usuarios con contraseña hasheada (nunca en texto plano).
- Login que emite un JWT con el rol del usuario.
- Perfil protegido con Bearer para cualquier usuario autenticado.
- Panel administrativo solo para rol admin; usuario normal recibe 403.
- Sin token se responde 401 y las contraseñas nunca aparecen en las respuestas.

> **Pista conceptual:** Ordena el flujo como lo vive el cliente: registro, login, token, perfil, panel. Cada paso debe poder probarse de forma independiente.

## 📦 Entregable

Cuaderno en verde, bitácora `auditorias/auditoria_sesion20.md` y tag `v0.2-hito2` en tu repositorio.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [2]:

%%writefile app.py
from fastapi import FastAPI, HTTPException, Depends
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from pydantic import BaseModel
from passlib.context import CryptContext
from jose import jwt, JWTError
from datetime import datetime, timedelta

app = FastAPI(title="API de Libros - Hito 2")

# Configuracion de seguridad
SECRET_KEY = "clave_secreta_hito2"
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")
security = HTTPBearer(auto_error=False)

# Base de datos temporal de usuarios
usuarios = {
    "admin": {
        "usuario": "admin",
        "password": pwd_context.hash("admin123"),
        "rol": "admin"
    }
}

# Modelo para recibir los datos
class UsuarioRegistro(BaseModel):
    usuario: str
    password: str

class UsuarioLogin(BaseModel):
    usuario: str
    password: str

# Funcion para proteger las contrasenas
def verificar_password(password, password_hash):
    return pwd_context.verify(password, password_hash)

# Funcion para crear el token JWT
def crear_token(usuario, rol):
    datos = {
        "sub": usuario,
        "rol": rol,
        "exp": datetime.utcnow() + timedelta(
            minutes=ACCESS_TOKEN_EXPIRE_MINUTES
        )
    }
    return jwt.encode(datos, SECRET_KEY, algorithm=ALGORITHM)

# Funcion para obtener al usuario autenticado
def obtener_usuario(
    credentials: HTTPAuthorizationCredentials = Depends(security)
):
    if credentials is None:
        raise HTTPException(status_code=401, detail="Token requerido")

    try:
        payload = jwt.decode(
            credentials.credentials,
            SECRET_KEY,
            algorithms=[ALGORITHM]
        )
        nombre = payload.get("sub")
        rol = payload.get("rol")

        if nombre is None or nombre not in usuarios:
            raise HTTPException(status_code=401, detail="Token invalido")

        return {"usuario": nombre, "rol": rol}

    except JWTError:
        raise HTTPException(status_code=401, detail="Token invalido")

# Ruta de registro
@app.post("/api/registro", status_code=201)
def registrar_usuario(datos: UsuarioRegistro):
    if datos.usuario in usuarios:
        raise HTTPException(status_code=400, detail="El usuario ya existe")

    usuarios[datos.usuario] = {
        "usuario": datos.usuario,
        "password": pwd_context.hash(datos.password),
        "rol": "user"
    }

    return {"mensaje": "Usuario registrado correctamente",
            "usuario": datos.usuario,
            "rol": "user"}

# Ruta de inicio de sesion
@app.post("/api/token")
def iniciar_sesion(datos: UsuarioLogin):
    usuario = usuarios.get(datos.usuario)

    if not usuario or not verificar_password(
        datos.password, usuario["password"]
    ):
        raise HTTPException(status_code=401, detail="Credenciales incorrectas")

    token = crear_token(usuario["usuario"], usuario["rol"])

    return {"access_token": token, "token_type": "bearer"}

# Ruta del perfil
@app.get("/api/perfil")
def consultar_perfil(usuario=Depends(obtener_usuario)):
    return {
        "usuario": usuario["usuario"],
        "rol": usuario["rol"]
    }

# Ruta exclusiva para administradores
@app.get("/api/admin/panel")
def panel_administrador(usuario=Depends(obtener_usuario)):
    if usuario["rol"] != "admin":
        raise HTTPException(
            status_code=403,
            detail="No tienes permisos de administrador"
        )

    return {
        "mensaje": "Bienvenido al panel administrativo",
        "usuario": usuario["usuario"],
        "rol": usuario["rol"]
    }


Writing app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Se trabajará con Python y FastAPI en Google Colab. Se utilizarán herramientas como JWT para la autenticación, bcrypt para proteger las contraseñas y HTTPX para realizar las pruebas de la API. El proyecto consiste en desarrollar una API de libros con usuarios y diferentes permisos. |
| **Rol** | Actuar como desarrollador de software con conocimientos en Python, FastAPI y seguridad de aplicaciones web, ayudando a implementar una API segura y funcional. |
| **Especificación** | La API debe cumplir con las siguientes funciones:

Registrar usuarios y guardar sus contraseñas mediante un hash.

Permitir que los usuarios inicien sesión con su nombre de usuario y contraseña.

Generar un token JWT que incluya el rol del usuario.

Permitir que los usuarios autenticados consulten su perfil.

Restringir el panel administrativo únicamente al usuario con rol de administrador.

Responder con código HTTP 401 cuando no exista una autenticación válida.

Responder con código HTTP 403 cuando un usuario normal intente entrar al panel administrativo.

Evitar que las contraseñas aparezcan en las respuestas. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [3]:
from verificar import probar_http

casos = [{'descripcion': 'Registro sin filtrar contraseñas',
  'metodo': 'POST',
  'ruta': '/api/registro',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 201,
  'no_contiene': ['password', 'hash']},
 {'descripcion': 'Login del usuario registrado',
  'metodo': 'POST',
  'ruta': '/api/token',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 200,
  'guardar': {'token_user': 'access_token'}},
 {'descripcion': 'El usuario accede a su perfil',
  'metodo': 'GET',
  'ruta': '/api/perfil',
  'headers': {'Authorization': 'Bearer {token_user}'},
  'estado': 200,
  'json_contiene': {'usuario': 'ana'}},
 {'descripcion': 'El usuario no entra al panel admin',
  'metodo': 'GET',
  'ruta': '/api/admin/panel',
  'headers': {'Authorization': 'Bearer {token_user}'},
  'estado': 403},
 {'descripcion': 'El administrador obtiene su token',
  'metodo': 'POST',
  'ruta': '/api/token',
  'json': {'usuario': 'admin', 'password': 'admin123'},
  'estado': 200,
  'guardar': {'token_admin': 'access_token'}},
 {'descripcion': 'El administrador entra al panel',
  'metodo': 'GET',
  'ruta': '/api/admin/panel',
  'headers': {'Authorization': 'Bearer {token_admin}'},
  'estado': 200},
 {'descripcion': 'Sin token no hay acceso', 'metodo': 'GET', 'ruta': '/api/perfil', 'estado': 401}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 7 casos · app.py
✅ 01. Registro sin filtrar contraseñas
✅ 02. Login del usuario registrado
✅ 03. El usuario accede a su perfil
✅ 04. El usuario no entra al panel admin
✅ 05. El administrador obtiene su token
✅ 06. El administrador entra al panel
✅ 07. Sin token no hay acceso
------------------------------------------------------------------------------
RESULTADO: 7/7 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion20.md` a `auditorias/auditoria_sesion20.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   Lo que más me costó fue lograr que el inicio de sesión funcionara correctamente y que se generara el token JWT, porque tuve que entender cómo se verifican las contraseñas y cómo se utiliza el token para permitir el acceso a las rutas protegidas.
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   Decidí separar las funciones de registro, inicio de sesión y verificación del token para tener el código más ordenado y que fuera más fácil identificar los errores. También asigné automáticamente el rol de usuario normal a las personas que se registran.
   - ¿Qué error detectó la verificación que tú no habías previsto?
   La verificación me ayudó a identificar que no era suficiente con que las rutas funcionaran, también tenía que comprobar que cada usuario tuviera los permisos correctos. Por ejemplo, un usuario normal no debía poder entrar al panel de administrador y una persona sin token no debía tener acceso al perfil.
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
